[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skyexry/urban-mobility-forecast/blob/main/notebooks/01b_station_selection.ipynb)

# 01b — Station Selection
Load preprocessed demand data, apply filters, remove isolated nodes, save final station list.

In [ ]:
!pip install folium -q

In [ ]:
import pandas as pd
import numpy as np
import folium
from sklearn.metrics.pairwise import haversine_distances
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Load data

In [ ]:
df = pd.read_parquet('/content/drive/MyDrive/citibike/hourly_demand_final.parquet')
df['hour'] = pd.to_datetime(df['hour'])
print(f'Shape      : {df.shape}')
print(f'Stations   : {df["start_station_id"].nunique()}')
print(f'Date range : {df["hour"].min()} → {df["hour"].max()}')

Shape      : (5956935, 6)
Stations   : 438
Date range : 2024-01-01 00:00:00 → 2025-12-31 23:00:00


## 2. Aggregate station stats

In [ ]:
total_hours = df['hour'].nunique()

stations = (
    df.groupby('start_station_id')
    .agg(
        name=('start_station_name', 'first'),
        lat=('start_lat', 'mean'),
        lng=('start_lng', 'mean'),
        total_demand=('demand', 'sum'),
        mean_hourly=('demand', 'mean'),
        hours_active=('hour', 'nunique'),
    )
    .reset_index()
)
stations['coverage'] = stations['hours_active'] / total_hours
print(f'Total stations: {len(stations)}, total hours: {total_hours}')
stations.sort_values('total_demand', ascending=False).head(10)

Total stations: 438, total hours: 16813


,start_station_id,name,lat,lng,total_demand,mean_hourly,hours_active,coverage
256,6140.05,W 21 St & 6 Ave,40.741740,-73.994156,447758,30.050872,14900,0.886219
197,5788.13,Lafayette St & E 8 St,40.730207,-73.991026,388866,26.594584,14622,0.869684
89,5329.03,West St & Chambers St,40.717548,-74.013221,375818,27.580948,13626,0.810444
287,6331.01,W 31 St & 7 Ave,40.749156,-73.991600,364742,23.483260,15532,0.923809
216,5905.14,University Pl & E 14 St,40.734814,-73.992085,360997,24.055241,15007,0.892583
215,5905.12,Broadway & E 14 St,40.734546,-73.990741,357770,23.562302,15184,0.903111
340,6726.01,11 Ave & W 41 St,40.760301,-73.998842,348662,23.074917,15110,0.898709
310,6492.08,9 Ave & W 33 St,40.752568,-73.996765,347526,23.945842,14513,0.863201
371,6948.10,Broadway & W 58 St,40.766953,-73.981693,337082,22.927629,14702,0.874442
303,6450.05,8 Ave & W 31 St,40.750585,-73.994685,333070,21.745120,15317,0.911021


## 3. Map — all 438 stations

In [ ]:
m = folium.Map(location=[stations['lat'].mean(), stations['lng'].mean()], zoom_start=12)
for _, row in stations.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lng']],
        radius=4, color='steelblue', fill=True, fill_opacity=0.7,
        tooltip=f"{row['name']} | demand={row['total_demand']:,.0f}",
    ).add_to(m)
m

## 4. Demand filter — top N by total demand (independent of region)

In [ ]:
DEMAND_THRESHOLD = stations['mean_hourly'].quantile(0.15)
print(f'Demand threshold (15th pct): {DEMAND_THRESHOLD:.3f} trips/hour')

stations_filtered = stations[stations['mean_hourly'] >= DEMAND_THRESHOLD].copy()
print(f'Stations after percentile filter: {len(stations_filtered)}')

TOP_N = 104  # slightly over 100 to account for isolated node removal
stations_final = (
    stations_filtered
    .sort_values('total_demand', ascending=False)
    .head(TOP_N)
    .copy()
)
print(f'Stations after top-{TOP_N} cap: {len(stations_final)}')

Demand threshold (15th pct): 8.190 trips/hour
Stations after percentile filter: 372
Stations after top-104 cap: 104


## 5. Isolated node detection & removal
Drop stations with no neighbor within the distance threshold — they would be isolated in the graph.

In [ ]:
ISOLATION_THRESHOLD_KM = 0.5

coords_rad = np.radians(stations_final[['lat', 'lng']].values)
dist_km = haversine_distances(coords_rad) * 6371
np.fill_diagonal(dist_km, np.inf)

stations_final = stations_final.copy()
stations_final['min_neighbor_km'] = dist_km.min(axis=1)

isolated_mask = stations_final['min_neighbor_km'] > ISOLATION_THRESHOLD_KM
print(f'Isolation threshold : {ISOLATION_THRESHOLD_KM} km')
print(f'Isolated stations   : {isolated_mask.sum()}')
if isolated_mask.any():
    print(stations_final[isolated_mask][['start_station_id', 'name', 'lat', 'lng', 'min_neighbor_km']].to_string(index=False))

stations_final = stations_final[~isolated_mask].copy()
print(f'\nFinal station count : {len(stations_final)}')

Isolation threshold : 0.5 km
Isolated stations   : 4
start_station_id                        name       lat        lng  min_neighbor_km
         4395.07      Hanson Pl & Ashland Pl 40.685068 -73.977908         3.251667
         5204.05          S 4 St & Wythe Ave 40.712859 -73.965903         0.551664
         5696.03 Pier 40 - Hudson River Park 40.727714 -74.011296         0.677264
         5294.04         Henry St & Grand St 40.714211 -73.981095         0.523313

Final station count : 100


## 6. Map — final stations

In [ ]:
m3 = folium.Map(location=[stations_final['lat'].mean(), stations_final['lng'].mean()], zoom_start=13)
for _, row in stations_final.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lng']],
        radius=5, color='darkorange', fill=True, fill_opacity=0.85,
        tooltip=f"{row['name']} | demand={row['total_demand']:,.0f} | avg={row['mean_hourly']:.1f}/hr",
    ).add_to(m3)
m3

## 7. Save

In [ ]:
final_ids = stations_final['start_station_id'].tolist()
df_final = df[df['start_station_id'].isin(final_ids)].copy()

print(f'Final stations : {len(stations_final)}')
print(f'Date range     : {df_final["hour"].min()} → {df_final["hour"].max()}')
print(f'Total rows     : {len(df_final):,}')

stations_final.to_parquet('/content/drive/MyDrive/citibike/stations_final.parquet', index=False)
df_final.to_parquet('/content/drive/MyDrive/citibike/hourly_demand_filtered.parquet', index=False)
print('Saved.')

Final stations : 100
Date range     : 2024-01-01 00:00:00 → 2025-12-31 23:00:00
Total rows     : 1,435,803
Saved.
